# Countdown: GRPO и reward hacking

Перед запуском выберите **Runtime → Change runtime type → T4 GPU**. Адрес репозитория уже указан ниже. Сравниваются strict и naive награды при одинаковых Qwen3-0.6B, LoRA, seed и бюджете. Сначала выполните `smoke`, затем `deadline` (50 шагов) или `full` (300 шагов).

In [ ]:
REPO_URL = 'https://github.com/AlexBuQA/SessionLern.git'  # @param {type:'string'}
from pathlib import Path
repo_name = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
repo_root = Path('/content') / repo_name
if not repo_root.exists():
    !git clone $REPO_URL $repo_root
%cd $repo_root/track3_countdown_grpo

In [ ]:
%pip install -q -r requirements.txt
%pip uninstall -y torchaudio torchao
import subprocess
import sys
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'countdown_rl.data', '--config', 'configs/full.json'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

## Сначала короткая проверка, затем эксперимент
Smoke запускает 2 шага и 8 задач и проверяет только память и совместимость. Для сдачи в ограниченный срок установите `MODE='deadline'`: 2 условия × 3 seed × 50 шагов, оценка каждые 25 шагов на 240 отложенных задачах. Вариант `full` сохраняет исходные 300 шагов.

In [ ]:
import subprocess
import sys
MODE = 'smoke'  # @param ['smoke', 'deadline', 'full']
assert MODE in {'smoke', 'deadline', 'full'}
command = [sys.executable, 'run.py', '--config', f'configs/{MODE}.json', '--output', f'results/{MODE}', '--outputs', f'outputs/{MODE}']
print('Running:', ' '.join(command), flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
returncode = process.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, command)

In [ ]:
from IPython.display import display, Image, Markdown
results = Path('results') / MODE
display(Markdown((results / 'RESULTS.md').read_text(encoding='utf-8')))
for name in ['training_reward.png', 'heldout_accuracy.png', 'kl_divergence.png', 'response_length.png']:
    if (results / name).exists():
        display(Image(filename=str(results / name)))

In [ ]:
import shutil
archive = shutil.make_archive(f'/content/countdown_{MODE}_results', 'zip', results)
print('Archive:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass